# Laboratorio 4: Playwright Dinámico

Hasta el momento, extrajimos datos utilizando la librería `requests` y analizando el código fuente con Beautiful Soup o Scrapling. Sin embargo, ¿qué sucede cuando la información que necesitamos no está en el HTML inicial, sino que la página utiliza JavaScript para dibujar la pantalla, despliega pop-ups molestos o bloquea a los bots de plano?

Entramos al territorio de la **Automatización de Navegadores**. En lugar de descargar el HTML a ciegas, levantaremos un navegador Google Chrome (Chromium) real, lo abriremos en nuestra pantalla, y le daremos instrucciones precisas simulando ser un ser humano.

Nuestra herramienta será **Playwright**, un marco de automatización moderno (creado por Microsoft) superior a tecnologías clásicas como Selenium.

> [!IMPORTANT]
> Playwright no puede ejecutarse directamente dentro de las celdas de Jupyter debido a un conflicto técnico con el bucle de eventos (`asyncio`) que Jupyter ya mantiene corriendo internamente. La solución estándar consiste en escribir nuestros scripts de Playwright en archivos `.py` externos usando el comando mágico `%%writefile`, y luego ejecutarlos desde la terminal con `!python`. Este patrón lo utilizaremos a lo largo de todo el laboratorio.

## 1. El Cambio de Paradigma: Browser, Context y Page

Playwright maneja una estructura jerárquica muy estricta para garantizar que las sesiones de navegación sean seguras y no guarden *cookies* (rastros) no deseadas entre ejecuciones:

1.  **Browser (Navegador):** Levantamos el motor de Chromium. Puede ser *Headless* (invisible) o *Headed* (visible para nosotros).
2.  **Context (Contexto):** Dentro del navegador, creamos un contexto de navegación. Pensemos en esto como abrir una **ventana de incógnito** completamente virgen.
3.  **Page (Página):** Dentro del contexto o ventana, abrimos finalmente una pestaña específica donde cargaremos el sitio web.

Llevaremos a cabo nuestro primer acercamiento interactivo abriendo la portada del diario La Nación.

In [ ]:
%%writefile lanacion_hola_mundo.py
import time
from playwright.sync_api import sync_playwright

with sync_playwright() as p:
    # 1. Instanciamos el navegador Chromium de forma visible (headless=False)
    print("Levantando el navegador...")
    browser = p.chromium.launch(headless=False)
    
    # 2. Creamos el contexto (modo incógnito puro)
    context = browser.new_context()
    
    # 3. Abrimos una pestaña nueva en ese contexto
    page = context.new_page()
    
    # Instruimos a la página a navegar a la URL
    print("Navegando hacia lanacion.com.ar...")
    page.goto("https://www.lanacion.com.ar/")
    
    # Esperamos 5 segundos reales para observar qué pasa
    time.sleep(5)
    
    # Obtenemos el título de la pestaña actual para confirmar que cargó
    print(f"Conexión exitosa. Título de la pestaña cargada: {page.title()}")
    
    # Cerramos el navegador para no saturar la memoria RAM de nuestra máquina
    browser.close()
    print("Navegador cerrado de forma segura.")

In [ ]:
# Ejecutamos el script externamente, por fuera del loop de Jupyter
!python lanacion_hola_mundo.py

> [!NOTE]
> Al ejecutar la celda anterior, el navegador se inicializó de manera visible. Se observará que el portal demora unos instantes en componer gráficamente todos sus bloques y fotos. Si usáramos `requests` clásico sin Playwright, la mitad de las secciones no existirían porque no ejecutarían su código JavaScript interno.

## 2. Interactividad: Evadir Modales en Sitios Molestos

Los portales noticiosos frecuentemente interrumpen la lectura desplegando carteles de gran tamaño (banners) invitándonos a aceptar *Cookies* o suscribirnos a boletines de correo (Newsletters).

Para un bot de extracción masiva, un Pop-Up gigante bloqueando la pantalla significa no poder ver ni extraer los títulos. Debemos enseñarle a **buscar** y **cerrar** esos modales.

Playwright usa **Localizadores (Locators)** para encontrar elementos en el Documento (DOM). En lugar de procesar HTML crudo, interactuamos con elementos vivos.

In [ ]:
%%writefile lanacion_modal.py
from playwright.sync_api import sync_playwright

with sync_playwright() as p:
    browser = p.chromium.launch(headless=False) # Lo mantenemos visible
    context = browser.new_context()
    page = context.new_page()
    
    print("Navegando a La Nación...")
    page.goto("https://www.lanacion.com.ar/")
    
    # Al ingresar a determinados diarios es común ver modales. 
    # Vamos a crear una táctica de "Intento de cierre" utilizando bloque try-except.
    # Localizamos el típico botón para Aceptar/Cerrar la política de privacidad.
    
    try:
        # wait_for_selector pausa la ejecución SIN USAR time.sleep hasta que el objeto exista en pantalla
        boton_aceptar = page.wait_for_selector('button:has-text("Aceptar")', timeout=5000)
        if boton_aceptar:
            boton_aceptar.click()
            print("Cuadro de diálogo de cookies identificado y cerrado exitosamente.")
    except Exception as e:
        print("No apareció banner en el plazo de espera de 5 segundos, la vía está libre.")
        
    browser.close()
    print("Navegador cerrado.")

In [ ]:
!python lanacion_modal.py

## 3. Extrayendo la Agenda Informativa de la Portada

Habiendo sorteado los cuadros emergentes, procederemos a capturar el núcleo de la actividad: **La Agenda Periodística del Día**.

La Nación, como la mayoría de los diarios modernos, organiza su portada visualmente en **bloques seccionales**: un encabezado `<h3>` indica la sección ("Política y economía", "Deportes", etc.) y, debajo de él, se agrupan los contenedores `<article>` con las noticias correspondientes. Esto presenta un desafío técnico: la etiqueta de sección (`<h3>`) **no está dentro** del artículo, sino **antes** del bloque entero.

### Desglosando la Estrategia (page.evaluate)

En lugar de forzar a Playwright a saltar hacia atrás buscando el título de cada bloque, la solución óptima es inyectar un *script* directamente en el navegador utilizando la función `page.evaluate()`. Al delegar este trabajo al motor JavaScript del cliente, logramos un procesamiento instantáneo.

El algoritmo que utilizaremos funciona como una **banda transportadora secuencial**:

1. **`querySelectorAll('h3, article')`**: Le pedimos al navegador que seleccione *todos* los encabezados y artículos de la página estrictamente en el orden en que aparecen de arriba hacia abajo.
2. **Memoria de Estado (`seccionActual`)**: Comenzamos asumiendo la sección "General".
3. **Iteración Secuencial (`for`)**: 
    *   Si el algoritmo se topa con un encabezado `<h3>`, actualiza su Memoria de Estado con ese nuevo nombre (ej: pasa a ser "Economía").
    *   Si el algoritmo se topa con un `<article>`, lee su título (`<h1>` o `<h2>`) y lo guarda adjuntándole la sección que tiene en la Memoria de Estado actualmente.

Este patrón documenta fielmente la jerarquía plana del DOM web y nos permite estructurar los datos perfectamente en un JSON.

In [ ]:
%%writefile lanacion_extraccion.py
import json
from playwright.sync_api import sync_playwright

with sync_playwright() as p:
    browser = p.chromium.launch(headless=False)
    context = browser.new_context()
    page = context.new_page()
    
    print("Inicializando sesión en La Nación web y sorteando bloqueos preliminares...")
    page.goto("https://www.lanacion.com.ar/")
    
    # Minimizamos avisos de pantalla
    try:
        boton = page.wait_for_selector('button:has-text("Aceptar")', timeout=3000)
        if boton: boton.click()
    except:
        pass
    
    # Esperamos que cargue la portada
    print("Esperando confirmación del renderizado...")
    page.wait_for_selector("article", timeout=10000)
    page.wait_for_timeout(2000)  # Damos tiempo extra para carga diferida
    
    # =====================================================================
    # ESTRATEGIA: La Nación organiza su portada en BLOQUES SECCIONALES.
    # Cada bloque tiene un encabezado <h3> con la sección (ej: "Política"),
    # seguido de varios <article> con noticias de esa sección.
    # El <h3> NO está DENTRO del <article>, sino ANTES de ellos.
    # Por eso, usamos JavaScript para recorrer el DOM secuencialmente,
    # asignando a cada artículo la última sección encontrada.
    # =====================================================================
    
    print("Recolectando artículos con contexto seccional...")
    
    articulos_extraidos = page.evaluate("""
    () => {
        const resultados = [];
        let seccionActual = "General";
        
        // Recorremos TODOS los elementos del body en orden de aparición
        const todos = document.querySelectorAll('h3, article');
        
        for (const nodo of todos) {
            // Si encontramos un h3 de sección, actualizamos la sección activa
            if (nodo.tagName === 'H3') {
                const texto = nodo.innerText.trim();
                if (texto && texto.length < 60) {
                    seccionActual = texto;
                }
            }
            
            // Si encontramos un article, extraemos su título
            if (nodo.tagName === 'ARTICLE') {
                const h2 = nodo.querySelector('h2');
                const h1 = nodo.querySelector('h1');
                const tituloEl = h1 || h2;
                
                if (tituloEl) {
                    const titulo = tituloEl.innerText.trim();
                    if (titulo && titulo.length > 5) {
                        resultados.push({
                            seccion: seccionActual,
                            titulo: titulo
                        });
                    }
                }
            }
        }
        return resultados;
    }
    """)
    
    print(f"Se recolectaron {len(articulos_extraidos)} piezas informativas.")
    browser.close()

# Guardamos el corpus
with open("lanacion_portada.json", "w", encoding="utf-8") as f:
    json.dump(articulos_extraidos, f, indent=4, ensure_ascii=False)

print("Archivo 'lanacion_portada.json' guardado correctamente.")

# Mostramos las primeras 5 noticias con su sección
for noticia in articulos_extraidos[:5]:
    print(f"[{noticia['seccion'].upper()}] -> {noticia['titulo']}")

In [ ]:
# Ejecutamos el script de extracción completo. Se abrirá una ventana de Chromium.
!python lanacion_extraccion.py

## 4. Verificación del Corpus Extraído

A diferencia de los scripts anteriores (que corrían fuera de Jupyter), cargar y leer un archivo JSON sí podemos hacerlo directamente en una celda de Jupyter sin problemas, ya que es puro Python clásico.

In [ ]:
import pandas as pd
import json

with open("lanacion_portada.json", "r", encoding="utf-8") as f:
    datos = json.load(f)

df = pd.DataFrame(datos)
print(f"El corpus contiene {len(df)} artículos de la portada.")
display(df.head())

## 5. Limpieza de Archivos Temporales

Con la integridad de la extracción verificada, procedemos a purgar los scripts auxiliares que generamos con `%%writefile`.

In [ ]:
import os
for archivo in ["lanacion_hola_mundo.py", "lanacion_modal.py", "lanacion_extraccion.py"]:
    if os.path.exists(archivo):
        os.remove(archivo)
        print(f"Eliminado: {archivo}")
print("Purga de entorno finalizada.")

---
# Consolidación y Repaso

## Glosario Acotado
*   **Browser (Navegador):** Instancia superior del subproceso informático en Playwright. Ejecuta las lógicas subyacentes del Chromium/Webkit real pero a través de terminales en memoria, y debe destruirse luego de finalizar para no generar fugas de memoria RAM (*Memory Leaks*).
*   **Browser Context:** Un ecosistema hermético dentro de la jerarquía de Playwright que separa la sesión de cualquier *cookie* acumulada por el Browser. Cumple un rol homólogo al de lanzar una ventana de Navegación de Incógnito completamente limpia.
*   **Headed vs Headless Mode:** Modalidades de visualización del *Browser*. *Headed* dibuja físicamente los gráficos de la página frente a la pantalla del analista. El *Headless* (sin cabeza / invisible) procesa internamente los frames prescindiendo de la renderización gráfica, multiplicando drásticamente el rendimiento de ejecución lógico.
*   **Locator (Localizador):** Herramienta que vincula de manera explícita y auto-retornable el objeto seleccionado de Python a su equivalente *vivo* en la pantalla de la página cargada en el DOM. Permite interactuar (cliquear, extraer texto) conforme estas piezas aparezcan asincrónicamente en la ventana.
*   **`%%writefile`:** Comando mágico de Jupyter que escribe el contenido de la celda en un archivo `.py` externo, permitiendo ejecutar luego código de Playwright por fuera del *loop* de Jupyter con `!python archivo.py`.

## Preguntas de Autoevaluación

**1. ¿Por qué no podemos ejecutar Playwright directamente dentro de una celda de Jupyter?**
Jupyter mantiene internamente un bucle de eventos asíncrono (`asyncio`). Playwright (en su versión sincrónica) intenta crear su propio bucle, lo que genera un conflicto insalvable. La solución es escribir el script en un archivo `.py` externo con `%%writefile` y ejecutarlo como proceso independiente con `!python`, tal como se aborda en el Lab 005 con Gates Notes.

**2. ¿Por qué resulta de extrema importancia cerrar el navegador con `browser.close()` o utilizar bloques `with`?**
Ejecutar y mantener navegadores web son de los subprocesos de mayor impacto operativo en cualquier computadora. Utilizar las envolturas sintácticas garantizan, en caso de finalizaciones anticipadas o pausas bruscas, detener el ejecutable y liberar la memoria, minimizando el *Memory Leak* (Fuga de Memoria).

**3. ¿Cuál es la diferencia fundamental entre extraer datos con `requests` + Beautiful Soup vs. Playwright?**
El dúo clásico solo descarga el código HTML plano sin compilar. Playwright levanta un motor de renderizado completo que ejecuta JavaScript en tiempo real, interactúa con la página como lo haría un humano (clics, scroll, esperar carga) y opera contra el DOM final renderizado, no contra el código fuente crudo."
